# V2: Amazon-grid CUDA reproducer for Triad + TEOS-10

V1 passed on a tiny Cartesian grid. V2 restores the geometry and numerical components from the failing Amazon trial while retaining only one ocean timestep. It includes the 170 x 170 x 20 regional latitude-longitude grid, 5 x 5 x 4 halo, NumericalEarth bathymetry, immersed boundary, exponential vertical grid, CATKE vertical mixing, WENO, split-explicit free surface, TEOS-10 temperature/salinity buoyancy, and Triad diffusion.

It omits rivers, passive dye, sponges, atmosphere, radiation, output writers, plotting, and long integration. The first case uses credential-free analytic T/S fields. An optional ECCO case is provided at the end.

In [ ]:
ENV["CUDA_LAUNCH_BLOCKING"] = "1"
using Pkg
using NumericalEarth
using Oceananigans
using Oceananigans.Units
using CUDA
using Dates
using Oceananigans.TurbulenceClosures: TriadIsopycnalSkewSymmetricDiffusivity
CUDA.allowscalar(false)
println("Julia: ", VERSION)
Pkg.status(["Oceananigans", "NumericalEarth", "CUDA", "CUDACore", "SeawaterPolynomials"])
println("CUDA.functional() = ", CUDA.functional())
@assert CUDA.functional()

## Exact Amazon regional grid and bathymetry

In [ ]:
arch = GPU()
Nx, Ny, Nz = 170, 170, 20
depth = 4000meters
z = ExponentialDiscretization(Nz, -depth, 0; scale=depth/4, mutable=false)
underlying_grid = LatitudeLongitudeGrid(arch;
    size=(Nx, Ny, Nz), halo=(5, 5, 4),
    longitude=(-58.5, -41.5), latitude=(-7.8, 9.2), z,
    topology=(Bounded, Bounded, Bounded))
bottom_height = regrid_bathymetry(underlying_grid;
    minimum_depth=10, interpolation_passes=10, major_basins=1)
grid = ImmersedBoundaryGrid(underlying_grid, GridFittedBottom(bottom_height);
    active_cells_map=true)
grid

## Original ocean numerical stack, without dye or forcing

In [ ]:
triad = TriadIsopycnalSkewSymmetricDiffusivity(κ_skew=1e3, κ_symmetric=1e3)
vertical_mixing = NumericalEarth.Oceans.default_ocean_closure()
free_surface = SplitExplicitFreeSurface(grid; substeps=70)
momentum_advection = WENOVectorInvariant(order=5)
tracer_advection = WENO(order=5)
ocean = ocean_simulation(grid;
    momentum_advection, tracer_advection, free_surface,
    closure=(triad, vertical_mixing),
    tracers=(:T, :S))
ocean.model

## Analytic initial T/S control

These fields remain in normal oceanographic ranges but contain horizontal and vertical density gradients. This tests whether the full geometry and numerical stack are sufficient without ECCO data.

In [ ]:
T0(λ, φ, z) = 20.0 - 0.35φ + 0.002z
S0(λ, φ, z) = 35.0 + 0.015φ - 0.0002z
set!(ocean.model, T=T0, S=S0)
Oceananigans.BoundaryConditions.fill_halo_regions!(ocean.model.tracers)

function report_TS(model, label)
    Ti, Si = Array(interior(model.tracers.T)), Array(interior(model.tracers.S))
    Ta, Sa = Array(parent(model.tracers.T)), Array(parent(model.tracers.S))
    println(label)
    println("  interior T: ", extrema(Ti), " finite=", all(isfinite, Ti))
    println("  interior S: ", extrema(Si), " finite=", all(isfinite, Si))
    println("  all storage T: ", extrema(Ta))
    println("  all storage S: ", extrema(Sa))
    println("  all storage S < -32: ", count(<(-32), Sa))
end
report_TS(ocean.model, "before analytic one-step case")

## Run exactly one original-size timestep

This may take a long time to compile. If a CUDA device exception occurs, save all output and restart Julia before running another GPU case.

In [ ]:
time_step!(ocean.model, 3minutes)
CUDA.synchronize()
report_TS(ocean.model, "after analytic one-step case")
println("PASS: full Amazon geometry/numerics with analytic T/S")

## Optional: exact ECCO initialization

Run this only in a fresh kernel after the analytic case passes. Set `ECCO_USERNAME` and `ECCO_PASSWORD` outside the notebook, rerun the setup/grid/model cells, then run this cell. Do not paste credentials into the notebook.

In [ ]:
@assert haskey(ENV, "ECCO_USERNAME") && haskey(ENV, "ECCO_PASSWORD")
date = DateTime(1993, 1, 1)
ecco_set = MetadataSet((:temperature, :salinity); dataset=ECCO4Monthly(), date)
set!(ocean.model, ecco_set)
Oceananigans.BoundaryConditions.fill_halo_regions!(ocean.model.tracers)
report_TS(ocean.model, "before ECCO one-step case")
time_step!(ocean.model, 3minutes)
CUDA.synchronize()
report_TS(ocean.model, "after ECCO one-step case")
println("PASS: full Amazon geometry/numerics with ECCO T/S")

## Interpretation

If the analytic case fails, the full Amazon geometry/numerical stack is sufficient. If analytic passes but ECCO fails, the trigger is tied to the initialized T/S data or its boundary/immersed values. If both pass, add the omitted model components one at a time; do not claim this notebook reproduces the issue.